given a certain layer l in the neural network:

Have a test set (x_i,y_i) and a disjoint concept set (x_j, c_j)
first compute the vector pointing in the direction of the concept v_c by fitting a linear regression to $v_C=(\Phi^l(x_j), c_j)$.

Then compute the activation of the layer in the test set $a=\frac{\partial f(x_i)}{\partial \Phi^l(x_i)}$

attain the sensitivity of layer l to the concept with $S^l_{C,i}=a \cdot v_C$


This results in a local measure of sensitivity. To attain a global measure define

$Br = R^2 \cdot \frac{\hat\mu}{\hat\sigma}$

with R^2 being the coefficient which determines how closely $v_C$ (the RCV) fits $\{\Phi^l(x_i),c_i\}^N_i$

The analysis will provide us with insight whether the network learns the provided concepts at all

Plan: Load model and functions acessing layer activation of the model

compute gradients of the output to the layer activation to attain a in a test set (train test split necessary?)

for the RCV, compute layer activation of test inputs  and use precomputed concepts (c_j) to fit a regression. direction given by RCV

In [ ]:
import pathlib
import random
import copy
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
import gc

import itertools
from scipy.stats import ttest_ind

import sklearn.model_selection
import sklearn.linear_model
import scipy.stats

#from act_max_util import *

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))


In [ ]:
class ActivationExtractor:
    def __init__(self, model):
        """
        Initialize the ActivationExtractor with a PyTorch model.
        
        Args:
            model: A PyTorch model (S4PatchedFinalNet in this case)
        """
        self.model = model
        self.activation = {}
        self.hooks = []
        
        # Map with full module paths
        self.method_map = {
            'trunk_net.spatial_filter.layers.conv_temporal': self.get_trunk_net_spatial_filter_layers_conv_temporal_activation,
            #'trunk_net.spatial_filter.layers.bnorm_temporal': self.get_trunk_net_spatial_filter_layers_bnorm_temporal_activation,
            'trunk_net.spatial_filter.layers.conv_spatial': self.get_trunk_net_spatial_filter_layers_conv_spatial_activation,
            #'trunk_net.spatial_filter.layers.pool_1': self.get_trunk_net_spatial_filter_layers_pool_1_activation,
            'trunk_net.spatial_filter.layers.conv_separable_depth': self.get_trunk_net_spatial_filter_layers_conv_separable_depth_activation,
            'trunk_net.spatial_filter.layers.conv_separable_point': self.get_trunk_net_spatial_filter_layers_conv_separable_point_activation,
            #'trunk_net.spatial_filter.layers.pool_2': self.get_trunk_net_spatial_filter_layers_pool_2_activation,
            #'trunk_net.conv1x1': self.get_trunk_net_conv1x1_activation,
            'trunk_net.s4_blocks.0': self.get_trunk_net_s4_blocks_0_activation,
            'trunk_net.spatial_filter': self.get_trunk_net_spatial_filter_activation,
            #'trunk_net.s4_blocks.0.post_tm_scale': self.get_trunk_net_s4_blocks_0_post_tm_scale_activation,
            #'trunk_net.s4_blocks.0.time_mixer.kernel': self.get_trunk_net_s4_blocks_0_time_mixer_kernel_activation,
            #'trunk_net.s4_blocks.0.time_mixer': self.get_trunk_net_s4_blocks_0_time_mixer_activation,
            #'trunk_net.s4_blocks.0.channel_mixer': self.get_trunk_net_s4_blocks_0_channel_mixer_activation,
            'cross_trial_s4': self.get_cross_trial_s4_activation,
            'head_net.fc_mean1': self.get_head_net_fc_mean1_activation
            
            #'cross_trial_s4.time_mixer': self.get_cross_trial_s4_time_mixer_activation,
            #'cross_trial_s4.channel_mixer': self.get_cross_trial_s4_channel_mixer_activation,
        }

    def get_activation(self, name):
        """
        Returns a hook function that captures the output of a layer.
        
        Args:
            name (str): Name to associate with the captured activation
            
        Returns:
            hook: A forward hook function
        """
        def hook(module, input, output):
            self.activation[name] = output
        return hook
    
    def register_hook(self, module_path, name=None):
        """
        Register a forward hook on a module specified by its path.
        
        Args:
            module_path (str): Dot-separated path to the module
            name (str, optional): Name for the activation. If None, uses module_path
            
        Returns:
            module: The module that the hook was registered on
        """
        if name is None:
            name = module_path
            
        parts = module_path.split('.')
        module = self.model
        
        # Navigate the module hierarchy
        for part in parts:
            if part.isdigit():  # Handle numeric indices for lists
                module = module[int(part)]
            else:
                module = getattr(module, part)
                
        # Register the hook
        hook = module.register_forward_hook(self.get_activation(name))
        self.hooks.append(hook)
        
        return module
    
    def clear_hooks(self):
        """Remove all registered hooks."""
        for hook in self.hooks:
            hook.remove()
        self.hooks = []
        
    def __call__(self, layer_name, input_data):
        """
        Get the activation for a specific layer.
        
        Args:
            layer_name (str): Full path of the layer in the model
            input_data: Input data to feed to the model
            
        Returns:
            The activation of the specified layer
        """
        if layer_name not in self.method_map:
            raise ValueError(f"Unknown layer name: {layer_name}")
        
        # Clear hooks and activations from previous calls
        self.clear_hooks()
        self.activation = {}
        
        return self.method_map[layer_name](input_data)

    def get_trunk_net_spatial_filter_layers_conv_temporal_activation(self, input_data):
        self.register_hook('trunk_net.spatial_filter.layers.conv_temporal', 'conv_temporal')
        output = self.model(input_data)
        return self.activation['conv_temporal']

    def get_trunk_net_spatial_filter_layers_bnorm_temporal_activation(self, input_data):
        self.register_hook('trunk_net.spatial_filter.layers.bnorm_temporal', 'bnorm_temporal')
        output = self.model(input_data)
        return self.activation['bnorm_temporal']

    def get_trunk_net_spatial_filter_layers_conv_spatial_activation(self, input_data):
        self.register_hook('trunk_net.spatial_filter.layers.conv_spatial', 'conv_spatial')
        output = self.model(input_data)
        return self.activation['conv_spatial']

    def get_trunk_net_spatial_filter_layers_pool_1_activation(self, input_data):
        self.register_hook('trunk_net.spatial_filter.layers.pool_1', 'pool_1')
        output = self.model(input_data)
        return self.activation['pool_1']

    def get_trunk_net_spatial_filter_layers_conv_separable_depth_activation(self, input_data):
        self.register_hook('trunk_net.spatial_filter.layers.conv_separable_depth', 'conv_separable_depth')
        output = self.model(input_data)
        return self.activation['conv_separable_depth']

    def get_trunk_net_spatial_filter_layers_conv_separable_point_activation(self, input_data):
        self.register_hook('trunk_net.spatial_filter.layers.conv_separable_point', 'conv_separable_point')
        output = self.model(input_data)
        return self.activation['conv_separable_point']

    def get_trunk_net_spatial_filter_layers_pool_2_activation(self, input_data):
        self.register_hook('trunk_net.spatial_filter.layers.pool_2', 'pool_2')
        output = self.model(input_data)
        return self.activation['pool_2']

    def get_trunk_net_conv1x1_activation(self, input_data):
        self.register_hook('trunk_net.conv1x1', 'conv1x1')
        output = self.model(input_data)
        return self.activation['conv1x1']
    
    def get_trunk_net_s4_blocks_0_activation(self, input_data):
        self.register_hook('trunk_net.s4_blocks.0', 'cross_timepoints')
        output = self.model(input_data)
        return self.activation['cross_timepoints']

    def get_trunk_net_s4_blocks_0_post_tm_scale_activation(self, input_data):
        self.register_hook('trunk_net.s4_blocks.0.post_tm_scale', 'post_tm_scale')
        output = self.model(input_data)
        return self.activation['post_tm_scale']

    def get_trunk_net_s4_blocks_0_time_mixer_kernel_activation(self, input_data):
        self.register_hook('trunk_net.s4_blocks.0.time_mixer.kernel', 'SSMKernelDPLR')
        output = self.model(input_data)
        return self.activation['SSMKernelDPLR']

    def get_trunk_net_s4_blocks_0_time_mixer_activation(self, input_data):
        self.register_hook('trunk_net.s4_blocks.0.time_mixer', 'time_mixer')
        output = self.model(input_data)
        return self.activation['time_mixer']

    def get_trunk_net_s4_blocks_0_channel_mixer_activation(self, input_data):
        self.register_hook('trunk_net.s4_blocks.0.channel_mixer', 'channel_mixer')
        output = self.model(input_data)
        return self.activation['channel_mixer']

    def get_cross_trial_s4_activation(self, input_data):
        self.register_hook('cross_trial_s4', 'cross_trial')
        output = self.model(input_data)
        return self.activation['cross_trial']

    def get_head_net_fc_mean1_activation(self, input_data):
        self.register_hook('head_net.fc_mean1', 'fc_mean1')
        output = self.model(input_data)
        return self.activation['fc_mean1']

    
    def get_trunk_net_spatial_filter_activation(self, input_data):
        self.register_hook('trunk_net.spatial_filter', 'spatial_filter')
        output = self.model(input_data)
        return self.activation['spatial_filter']
    
    def get_cross_trial_s4_time_mixer_activation(self, input_data):
        self.register_hook('cross_trial_s4.time_mixer', 'time_mixer')
        output = self.model(input_data)
        return self.activation['time_mixer']
    
    def get_cross_trial_s4_channel_mixer_activation(self, input_data):
        self.register_hook('cross_trial_s4.channel_mixer', 'channel_mixer')
        output = self.model(input_data)
        return self.activation['channel_mixer']
    
    def get_all_activations(self, input_data):
        """
        Get activations from all registered layers in a single forward pass.
        
        Args:
            input_data: Input data to feed to the model
            
        Returns:
            dict: Dictionary with activations for all layers in method_map
        """
        # Clear any existing hooks
        self.clear_hooks()
        self.activation = {}
        
        # Register hooks for all layers
        for layer_path in self.method_map.keys():
            self.register_hook(layer_path)
            
        # Forward pass
        output = self.model(input_data)
        
        # Store the results and clear hooks
        results = self.activation.copy()
        self.clear_hooks()
        
        return results

In [ ]:
def load_data_set(subject_index=2):
    cfg = load_config()
    cfg.dataset.subject_index =  subject_index
    all_epochs, labels_raw, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    print(all_epochs.shape)
    
    return all_epochs, labels_raw, ch_names

In [ ]:
def load_model(cfg, start_index=100, subject_index=2):
    save_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/model_checkpoints/finetune"
    
    file_path = os.path.join(save_path, f"subject_{subject_index}", f"model_checkpoint_finetune_subject_index_{subject_index}_start_idx_{start_index}_rep_0_pen.pth") 
    trunk_net = TrunkNet(n_chans=input_shape_st[0], n_times=input_shape_st[1])
    head_net = HeadNet(64, 1)  # Assuming these are the correct dimensions
    model = S4PatchedFinalNet(64, trunk_net, head_net)
    
    weights = torch.load(file_path)
    model.load_state_dict(weights)
    #model.load_state_dict(full_checkpoint['model_state_dict'])
    model.eval()
    model.to(device)
    return model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def py_ang(v1, v2):
    cos = np.dot(v1,v2)
    return np.arccos(cos/(np.linalg.norm(v1) * np.linalg.norm(v2)))

In [ ]:
def solve_regression(inputs, y, n_splits=3, n_repeats=1, random_state=12883823, verbose=0):
    scores=[]
    max_score = 0
    direction = None
    dirs=[]
    rkf = sklearn.model_selection.RepeatedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=random_state)
    counter = 0
    for train, test in rkf.split(inputs):
        if verbose:
            print('N. ', counter, '..')
            print(len(inputs[train]))
        reg = sklearn.linear_model.LinearRegression()
        reg.fit(inputs[train], y[train])
        trial_score = reg.score(inputs[test], y[test])
        #print('trial_score', trial_score)
        
        #print 'y[train]', y[train]
        # only if correlation found append results
        # append R2 scores (note that original only takes mean, but this may be unfair as max is 1 but unbounded below)
        scores.append(trial_score)
        if trial_score > max_score:
            # only take best fit direction
            direction = reg.coef_
            

            # store all directions for which coefficient is positive?
            dirs.append(reg.coef_)
        if verbose:
            print(trial_score)
        counter += 1
    if verbose:
        print(np.mean(scores))
        i=0
        while i+1<len(dirs):
            #print( 'angle: ', py_ang(dirs[i], dirs[i+1]))
            i+=1
    return max(scores),scores, direction, dirs

In [ ]:
cfg = load_config()
input_shape_st = (60, 900)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   

In [ ]:
def eeg_band_signal_generator(band_name, band_power, freq_bands, sampling_rate=1000, n_timepoints=900, random_seed=None):

    if random_seed is not None:
        np.random.seed(random_seed)
    
    # Generate frequency axis
    freqs = np.fft.rfftfreq(n_timepoints, d=1/sampling_rate)
    # Create a frequency domain signal (all zeros initially)
    fft_signal = np.zeros(len(freqs), dtype=complex)
    # Get the target band frequency range
    band_min, band_max = freq_bands[band_name]
    
    # Create a mask for the target frequency band
    band_mask = (freqs >= band_min) & (freqs <= band_max)

    n_bins = np.sum(band_mask)
    # Generate random phases for frequencies in the band
    phases = np.random.uniform(0, 2*np.pi, n_bins)
    amplitude = np.sqrt(2 * band_power / n_bins)
    fft_signal[band_mask] = amplitude * np.exp(1j * phases)

    time_signal = np.fft.irfft(fft_signal, n=n_timepoints)

    return time_signal

In [ ]:
def generate_random_concepts(subject_index, ch_idx, ch_name, band_name="alpha", concept="bandpower", n_samples=300):
    freq_bands = {
            "delta": (2, 4),
             "theta": (4, 8),
              "alpha": (8, 12),
              "beta": (13, 30),
              "gamma": (30, 45)}
    concept_dir ="/home/marco/Documents/GitHub/tms_eeg_decoding/TCAV"
    file_name = f"power_and_fractal_{subject_index}.csv"
    file_path = os.path.join(concept_dir, file_name)
    df = pd.read_csv(file_path)
    df = df[(df["channel_name"] == ch_name) & (df["frequency_band"] == band_name)]
    #min, max = df[concept].min(), df[concept].max()
    min, max = df[concept].quantile(0.02), df[concept].quantile(0.98)
    print(min, max)
    random_concepts_values = np.random.uniform(min, max, n_samples)
    random_signals = np.zeros((n_samples,60,900))

    for i in range(n_samples):
        band_power = random_concepts_values[i]
        signal = eeg_band_signal_generator(band_name, band_power, freq_bands)
        random_signals[i,ch_idx,:] = signal
    return random_signals, random_concepts_values

In [ ]:
def  load_subject_concepts(concept = "bandpower", channel="C4", freq_band="gamma", subject_index=2, n_samples=200):
    concept_dir ="/home/marco/Documents/GitHub/tms_eeg_decoding/TCAV"
    concept_vals = np.zeros(n_samples)
    data = np.zeros((n_samples, 60, 900))
    file_name = f"power_and_fractal_{subject_index}.csv"
    file_path = os.path.join(concept_dir, file_name)
    df = pd.read_csv(file_path)
    df = df[df["channel_name"] == channel]
    df = df[df["frequency_band"] == freq_band]
    # Take last 200 trials
    df = df.tail(n_samples)
    # Select only needed columns and store in a list
    df= df[[concept, 'trial_index']]
    trial_indices = df['trial_index'].values
    data = load_data_set(subject_index = subject_index)[0][trial_indices]
    # Filter data based on trial indices from df
    concept_vals = df[concept].values


    return data, concept_vals

In [ ]:
def get_model_regression_results(model, layer, data, concept_vals, act_extractor, normalize=False):

    layer_act = act_extractor(layer, data)
    #score2, direction2, dirs2 = solve_regression_pytorch(layer_act.view(layer_act.shape[0], -1), torch.tensor(concept_vals).to(device).float(), n_splits=3, n_repeats=1, random_state=12883823, verbose=0)
    if isinstance(layer_act, tuple):
        layer_act = layer_act[0]
        
    if layer == "cross_trial_s4":
        layer_act = layer_act.squeeze(0)
    layer_act = layer_act.cpu().detach().numpy()
    #print("pre", layer_act.shape)
    layer_act = layer_act.reshape(layer_act.shape[0], -1)
    #print("post", layer_act.shape)
    max_score, scores, best_dir, dirs = solve_regression(layer_act, concept_vals, n_splits=3, n_repeats=1, random_state=12883823, verbose=0)
    #score2, direction2, dirs2 = solve_regression_pytorch(layer_act, concept_vals, n_splits=3, n_repeats=1, random_state=12883823, verbose=0)
    #print("Scores", score, score2)
    if normalize:
        if best_dir is not None:
            best_dir = best_dir / np.linalg.norm(best_dir)
       
    return max_score, scores, best_dir, dirs

In [ ]:
def regress_activations_concepts(model, data, concept_vals, normalize=False):
    """
    Evaluate each layer of the model against concept values.
    
    Args:
        model: The neural network model
        data: Input data tensor
        concept_vals: Concept values to evaluate against
        
    Returns:
        dict: Dictionary with layer names as keys and their scores as values
    """
    layer_best_score = {}
    layer_best_direction = {}
    layer_scores = {}
    layer_direction = {}
    mean_score = {}
    d = torch.tensor(data.reshape(-1,60,900), dtype=torch.float32).to(device)
    act_extractor = ActivationExtractor(model)
    
    for layer in act_extractor.method_map.keys():
    
        #print(layer)
        max_scores, scores, best_dir, dirs = get_model_regression_results(model, layer, d, concept_vals.reshape(-1), act_extractor, normalize=normalize)
        layer_scores[layer] = scores
        layer_direction[layer] = dirs
        layer_best_score[layer] = max_scores
        layer_best_direction[layer] = best_dir
        mean_score[layer] = np.mean(scores)
        
    return layer_best_score, layer_scores, layer_best_direction, layer_direction, mean_score


In [ ]:
def derivative_output_layeractivation3(model, test_data, target_layer_name, normalize=False, absolute=False):
    model = model.eval()
    
    # Find the target layer
    target_layer = None
    for name, module in model.named_modules():
        if name == target_layer_name:
            target_layer = module
            break
    
    if target_layer is None:
        raise ValueError(f"Layer '{target_layer_name}' not found in model.")
    
    derivative_output_layer = []
    
    for data in test_data:
        # Clear activations for each sample
        activations = {}
        
        def save_activation(module, input, output):
            if isinstance(output, tuple):
                output = output[0]
            output.retain_grad()
            activations['output'] = output
        
   
        handle = target_layer.register_forward_hook(save_activation)
        
        # Ensure model is in fresh state
        model.zero_grad()
        torch.cuda.empty_cache() 
        
        input = torch.tensor(data.reshape(1, 60, 900), dtype=torch.float32, requires_grad=True).to(device)
        
        output = model(input)
        output[:, 0].backward(retain_graph=False)  # Changed to False if possible
        
        if 'output' in activations and activations['output'].grad is not None:
            if absolute and normalize:
                derivative_output_layer.append(np.abs(activations['output'].grad.reshape(-1).cpu().detach().numpy())/ torch.norm(activations['output'].grad).cpu().detach().numpy())
            
            
                #if normalize:
                #    derivative_output_layer.append(activations['output'].grad.reshape(-1).cpu().detach().numpy() / #torch.norm(activations['output'].grad).cpu().detach().numpy())
            
            elif absolute:
                derivative_output_layer.append(np.abs(activations['output'].grad.reshape(-1).cpu().detach().numpy()))
            elif normalize:
                derivative_output_layer.append(activations['output'].grad.reshape(-1).cpu().detach().numpy() / torch.norm(activations['output'].grad).cpu().detach().numpy())
            else:

                derivative_output_layer.append(activations['output'].grad.reshape(-1).cpu().detach().numpy())
        else:
            print(f"Warning: No gradient found for sample")
        
        # Remove hook immediately after use
        handle.remove()
        
        # Clear memory
        del input, output

        
    return np.stack(derivative_output_layer) if derivative_output_layer else None

In [ ]:
def compute_sensitivity(v_C, a):
    #extractor = ActivationExtractor(model)
    #layer_act = extractor(layer, test_data)
    #layer_act = act_extractor(layer, test_data)

    # Reshape v_C to match the batch dimension of a
    # v_C is (n_features,) and a is (batch_size, n_features)
    # We want to get (batch_size,) as output
    sensitivity = np.dot(a, v_C)
    return sensitivity

In [ ]:
def compute_BR_score(sensitivites, R_squared):
    mean_sensitive = np.mean(sensitivites)
    std_sensitive = np.std(sensitivites)
    Br = R_squared * (mean_sensitive/std_sensitive )
    return Br

# complete pipeline

## for subject 2

In [ ]:

# Things to try: Normaliue v_C too, also try non abs values.
bands= ["delta", "theta", "alpha", "beta", "gamma"]
layers=['trunk_net.spatial_filter.layers.conv_temporal', 'trunk_net.spatial_filter.layers.conv_spatial', 'trunk_net.spatial_filter.layers.conv_separable_point',
         'trunk_net.spatial_filter.layers.conv_separable_depth','trunk_net.spatial_filter' , 
        'trunk_net.s4_blocks.0', 'cross_trial_s4', 'head_net.fc_mean1']
ignore_channels = ['Cz', 'Iz', 'Fz', 'Oz', 'Pz']
subject_index = 2
_,_,ch_names = load_data_set(subject_index=subject_index)
model2_100 = load_model(cfg, start_index=100, subject_index=subject_index)
model2_200 = load_model(cfg, start_index=200, subject_index=subject_index)
model2_300 = load_model(cfg, start_index=300, subject_index=subject_index)
model2_400 = load_model(cfg, start_index=400, subject_index=subject_index)
model2_500 = load_model(cfg, start_index=500, subject_index=subject_index)
models = [model2_100, model2_200, model2_300, model2_400, model2_500]
n_samples = 250
for rep in [7,8,9]:
    results = {i : {band: {ch_name: {layer: None for layer in layers} for ch_name in ch_names} for band in bands} for i in np.arange(100,501,100)}
    for freq_band in bands:
        print("\nfreq_band\n", freq_band)
        for ch_idx, ch_name in tqdm(enumerate(ch_names)):
            if ch_name in ignore_channels:
                continue
            print("\nch_name\n", ch_name)
            data = load_subject_concepts(channel=ch_name, freq_band=freq_band, concept="bandpower", subject_index=subject_index, n_samples=n_samples)
            data_subject = data[0]
            concept_vals_subject = data[1]
            concept_data = generate_random_concepts(subject_index, ch_idx, ch_name,band_name=freq_band, concept="bandpower", n_samples=n_samples)
            #concept_data = load_random_concepts(channel=ch_name, freq_band=freq_band, concept="bandpower", subject_index=subject_index)
            data_random = concept_data[0]
            concept_vals_random = concept_data[1]
            max_score = {}
            scores = {}
            best_dir = {}
            dirs = {}
            mean_score = {}
            for model, idx in zip(models,np.arange(100,501,100)):
                max_score[idx], scores[idx], best_dir[idx], dirs[idx], mean_score[idx] = regress_activations_concepts(model, data_random, concept_vals_random, normalize=True)
    
            
            for layer in layers:
                for model, idx in zip(models,np.arange(100,501,100)):
                    if best_dir[idx][layer] is not None:
                        grad_activations = {idx: {layer : derivative_output_layeractivation3(model, data_subject, layer, absolute=False, normalize=False)}, }
                        # test why tf I get 500 instead of 250 sensitivites
                        sensitivites = compute_sensitivity(best_dir[idx][layer], grad_activations[idx][layer])
                        br = compute_BR_score(sensitivites, max_score[idx][layer])
                        results[idx][freq_band][ch_name][layer] = {"sensitivies" : sensitivites, "Br": br, "best_score": max_score[idx][layer], "scores" : scores[idx][layer], "mean_score": mean_score[idx][layer]}
            os.makedirs("results_channel_new2", exist_ok=True)
            np.save(f"results_channel_new2/RCAV_band_power_results_subject_{subject_index}_freq_band_{freq_band}_ch_name_{ch_name}_rep_{rep}.npy", results)
    os.makedirs("results_new_generator_norm_non_abs3", exist_ok=True)
    np.save(f"results_new_generator_norm_non_abs3/RCAV_band_power_results_subject_{subject_index}_{rep}.npy", results)

                #grad_activations = {idx: {layer : derivative_output_layeractivation3(model, data_subject, layer) for model, idx in zip(models,np.arange(100,501,100))}}

                #if dict_random_100_dirs[layer] is not None:
                #    sensitivites = compute_sensitivity(dict_random_100_dirs[layer], grad_activations_100)
                #    br = compute_BR_score(sensitivites_random_100, dict_random_100_scores[layer])
                #    results[freq_band][ch_name][layer] = {"Br: ": br_100_random, "sensitivities": sensitivites_random_100, "scores": dict_random_100_scores[layer]}
 



In [ ]:
bands= ["delta", "theta", "alpha", "beta", "gamma"]
layers=['trunk_net.spatial_filter.layers.conv_temporal', 'trunk_net.spatial_filter.layers.conv_spatial', 'trunk_net.spatial_filter.layers.conv_separable_point',
         'trunk_net.spatial_filter.layers.conv_separable_depth','trunk_net.spatial_filter' , 
        'trunk_net.s4_blocks.0', 'cross_trial_s4', 'head_net.fc_mean1']
ignore_channels = ['Cz', 'Iz', 'Fz', 'Oz', 'Pz']
subject_index = 2
_,_,ch_names = load_data_set(subject_index=subject_index)
model2_100 = load_model(cfg, start_index=100, subject_index=subject_index)
model2_200 = load_model(cfg, start_index=200, subject_index=subject_index)
model2_300 = load_model(cfg, start_index=200, subject_index=subject_index)
model2_400 = load_model(cfg, start_index=400, subject_index=subject_index)
model2_500 = load_model(cfg, start_index=500, subject_index=subject_index)
models = [model2_100, model2_200, model2_300, model2_400, model2_500]

for rep in range(10):
    results = {i : {band: {ch_name: {layer: None for layer in layers} for ch_name in ch_names} for band in bands} for i in np.arange(100,501,100)}
    for freq_band in bands:
        print("\nfreq_band\n", freq_band)
        for ch_name in tqdm(ch_names):
            if ch_name in ignore_channels:
                continue
            print("\nch_name\n", ch_name)
            data = load_subject_concepts(channel=ch_name, freq_band=freq_band, concept="bandpower", subject_index=subject_index)
            data_subject = data[0]
            concept_vals_subject = data[1]
            concept_data = load_random_concepts(channel=ch_name, freq_band=freq_band, concept="bandpower", subject_index=subject_index)
            data_random = concept_data[0]
            concept_vals_random = concept_data[1]
            random_scores = {}
            random_dirs = {}
            for model, idx in zip(models,np.arange(100,501,100)):
                random_scores[idx], random_dirs[idx] = regress_activations_concepts(model, data_random, concept_vals_random)
    
            for layer in layers:
                for model, idx in zip(models,np.arange(100,501,100)):
                    if random_dirs[idx][layer] is not None:
                        grad_activations = {idx: {layer : derivative_output_layeractivation3(model, data_subject, layer)}}
                        sensitivites = compute_sensitivity(random_dirs[idx][layer], grad_activations[idx][layer])
                        br = compute_BR_score(sensitivites, random_scores[idx][layer])
                        results[idx][freq_band][ch_name][layer] = {"Br: ": br, "sensitivities": sensitivites, "scores": random_scores[idx][layer]}
            os.makedirs("results_channel_new", exist_ok=True)
            np.save(f"results_channel/RCAV_band_power_results_subject_{subject_index}_freq_band_{freq_band}_ch_name_{ch_name}_rep_{rep}.npy", results)
    os.makedirs("results", exist_ok=True)
    np.save(f"results/RCAV_band_power_results_subject_{subject_index}_{rep}.npy", results)

                #grad_activations = {idx: {layer : derivative_output_layeractivation3(model, data_subject, layer) for model, idx in zip(models,np.arange(100,501,100))}}

                #if dict_random_100_dirs[layer] is not None:
                #    sensitivites = compute_sensitivity(dict_random_100_dirs[layer], grad_activations_100)
                #    br = compute_BR_score(sensitivites_random_100, dict_random_100_scores[layer])
                #    results[freq_band][ch_name][layer] = {"Br: ": br_100_random, "sensitivities": sensitivites_random_100, "scores": dict_random_100_scores[layer]}
 



# for all subjects

In [ ]:
model2_100

In [ ]:
cfg.dataset.test_subject_indices

In [ ]:
mne.set_log_level('ERROR')

In [ ]:
layers=['trunk_net.spatial_filter.layers.conv_temporal','trunk_net.spatial_filter.layers.conv_spatial', 'trunk_net.spatial_filter.layers.pool_1',
        'trunk_net.spatial_filter.layers.conv_separable_depth', 'trunk_net.spatial_filter.layers.conv_separable_point','trunk_net.spatial_filter',
         'trunk_net.conv1x1', 'trunk_net.s4_blocks.0.time_mixer', 'trunk_net.s4_blocks.0.channel_mixer', 'trunk_net.s4_blocks.0','cross_trial_s4.time_mixer', 'cross_trial_s4.channel_mixer', 'cross_trial_s4', 'head_net.fc_mean1']
ignore_channels = ['Cz', 'Iz', 'Fz', 'Oz', 'Pz']
for subject_index in cfg.dataset.test_subject_indices:
    _,_,ch_names = load_data_set(subject_index=subject_index)
    model2_100 = load_model(cfg, start_index=100, subject_index=subject_index)
    model2_400 = load_model(cfg, start_index=400, subject_index=subject_index)
    result_100 = {}
    result_400 = {}
    all_results_100 = {}
    all_results_400 = {}
    for freq_band in ["delta", "theta", "alpha", "beta", "gamma"]:
        result_100[freq_band] = {}
        result_400[freq_band] = {}
        all_results_100[freq_band] = {}
        all_results_400[freq_band] = {}
        print("\nfreq_band\n", freq_band)
        for ch_name in tqdm(ch_names):
            if ch_name in ignore_channels:
                continue
            print("\nch_name\n", ch_name)
            result_100[freq_band][ch_name] = {}
            result_400[freq_band][ch_name] = {}
            all_results_100[freq_band][ch_name] = {}
            all_results_400[freq_band][ch_name] = {}
            data = load_subject_concepts(channel=ch_name, freq_band=freq_band, concept="bandpower", subject_index=subject_index)
            data_subject = data[0]
            concept_vals_subject = data[1]
            concept_data = load_random_concepts(channel=ch_name, freq_band=freq_band, concept="bandpower", subject_index=subject_index)
            data_random = concept_data[0]
            concept_vals_random = concept_data[1]
            dict_random_100_scores, dict_random_100_dirs = regress_activations_concepts(model2_100, data_random, concept_vals_random)
            dict_random_400_scores, dict_random_400_dirs = regress_activations_concepts(model2_400, data_random, concept_vals_random)
            for layer in layers:
                print(layer)
                result_100[freq_band][ch_name][layer] = None
                result_400[freq_band][ch_name][layer] = None
                all_results_100[freq_band][ch_name][layer] = None
                all_results_400[freq_band][ch_name][layer] = None
                br_100_random = None
                br_400_random = None
                grad_activations_100 = derivative_output_layeractivation3(model2_100, data_subject, layer)
                grad_activations_400 = derivative_output_layeractivation3(model2_400, data_subject, layer)
                #print(dict_random_100_dirs[layer])
                if dict_random_100_dirs[layer] is not None:
                     sensitivites_random_100 = compute_sensitivity(dict_random_100_dirs[layer], grad_activations_100)
                     br_100_random = compute_BR_score(sensitivites_random_100, dict_random_100_scores[layer])
                     result_100[freq_band][ch_name][layer] = br_100_random
                if dict_random_400_dirs[layer] is not None:
                    sensitivites_random_400 = compute_sensitivity(dict_random_400_dirs[layer], grad_activations_400)
                    br_400_random = compute_BR_score(sensitivites_random_400, dict_random_400_scores[layer])
                    result_400[freq_band][ch_name][layer] = br_400_random

                dir = "random_intermediate_all_subjects"
                os.makedirs(dir, exist_ok=True)
                file_100 = os.path.join(dir, f"br_results_subject_{subject_index}_{freq_band}_{ch_name}_{layer}_100_ignore_channels.npy")
                file_400 = os.path.join(dir, f"br_results_subject_{subject_index}_{freq_band}_{ch_name}_{layer}_400_ignore_channels.npy")
                np.save(file_100, br_100_random, allow_pickle=True)
                np.save(file_400, br_400_random, allow_pickle=True)
                all_results_100[freq_band][ch_name][layer] = br_100_random
                all_results_400[freq_band][ch_name][layer] = br_400_random
        dir = "random_complete_all_subjects"
        os.makedirs(dir, exist_ok=True)
        file_100_all = os.path.join(dir, f"br_results_subject_{subject_index}_{freq_band}_100_ignore_channels.npy")
        file_400_all = os.path.join(dir, f"br_results_subject_{subject_index}_{freq_band}_400_ignore_channels.npy")
        np.save(file_100_all, all_results_100 , allow_pickle=True)
        np.save(file_400_all, all_results_400, allow_pickle=True)
        
